In [0]:
import pyspark.sql.functions as F

In [0]:
def silver_repo_data():
    endpoints = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
    for endpoint in endpoints:
        print(endpoint)

        repo_name = endpoint
        df = spark.read.format("delta").table(f"workspace.bronze.raw_{endpoint}_repo_data")
        cleaned_repo_data = df \
        .select(
            "id",
            F.trim(F.col("name")).alias("name"),
            F.trim(F.col("full_name")).alias("full_name"),
            F.col("organization.login").alias("org_login"),
            F.trim(F.col("description")).alias("description"),
            F.trim(F.col("language")).alias("language"),
            "created_at",
            "private",
            "fork",
            "is_template",
            "allow_forking",
            "has_issues",
            "has_pull_requests",
            "has_discussions",
            "has_wiki",
            "has_pages",
            "has_downloads",
            "pull_request_creation_policy",
            "size",
            "stargazers_count",
            "watchers_count",
            "forks_count",
            "network_count",
            "subscribers_count",
            "open_issues_count",
            F.col("license.name").alias("license_name"),
            "node_id"
        ) \
        .withColumn("created_at", F.to_timestamp("created_at")) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .fillna("n/a") \
        .dropDuplicates()

        cleaned_repo_data.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_repo_data")
        print(f"Saved: workspace.silver.{repo_name}_repo_data")

        print("=" * 50)

In [0]:
silver_repo_data()

In [0]:
def silver_pull_requests():
    endpoints = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
    for endpoint in endpoints:
        print(endpoint)
        repo_name = endpoint
        df = spark.read.format("delta").table(f"workspace.bronze.inc_{endpoint}_pull_requests")
        cleaned_pull_requests = df \
        .select(
            F.col("id").alias("pr_id"),
            "number",
            F.trim(F.col("title")).alias("title"),
            F.trim(F.col("state")).alias("state"),
            "draft",
            "locked",
            F.trim(F.col("body")).alias("body"),
            F.trim(F.col("author_association")).alias("author_association"),
            F.col("user.id").alias("user_id"),
            F.trim(F.col("user.login")).alias("user_login"),
            F.trim(F.col("user.type")).alias("user_type"),
            F.col("user.site_admin").alias("user_site_admin"),
            "created_at",
            "updated_at",
            "closed_at",
            "merged_at",
            F.trim(F.col("merge_commit_sha")).alias("merge_commit_sha"),
            F.trim(F.col("base.ref")).alias("base_ref"),
            F.trim(F.col("base.sha")).alias("base_sha"),
            F.col("base.repo.id").alias("base_repo_id"),
            F.trim(F.col("base.repo.full_name")).alias("base_repo_full_name"),
            F.trim(F.col("head.ref")).alias("head_ref"),
            F.trim(F.col("head.sha")).alias("head_sha"),
            F.col("head.repo.id").alias("head_repo_id"),
            F.trim(F.col("head.repo.full_name")).alias("head_repo_full_name"),
            F.col("milestone.id").alias("milestone_id"),
            F.trim(F.col("milestone.title")).alias("milestone_title"),
            F.trim(F.col("milestone.state")).alias("milestone_state"),
            F.trim(F.col("auto_merge.merge_method")).alias("auto_merge_method")
        ) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("created_at", F.to_timestamp("created_at")) \
        .withColumn("updated_at", F.to_timestamp("updated_at")) \
        .withColumn("closed_at", F.to_timestamp("closed_at")) \
        .withColumn("merged_at", F.to_timestamp("merged_at")) \
        .withColumn(
            "timestamp_issue",
            F.when(F.col("created_at") > F.col("updated_at"), F.lit("created_at after updated_at"))
            .when(F.col("closed_at") < F.col("created_at"), F.lit("closed_at before created_at"))
            .when(F.col("merged_at") < F.col("created_at"), F.lit("merged_at before created_at"))
            .when(F.col("merged_at") > F.col("closed_at"), F.lit("merged_at after closed_at"))
            .otherwise(F.lit("n/a"))
        ) \
        .withColumn("title", F.when(F.col("title").isNull(), F.lit("n/a")).otherwise(F.col("title"))) \
        .withColumn("state", F.when(F.col("state").isNull(), F.lit("n/a")).otherwise(F.col("state"))) \
        .withColumn("body", F.when(F.col("body").isNull() | (F.col("body") == ""), F.lit("n/a")).otherwise(F.col("body"))) \
        .withColumn("author_association", F.when(F.col("author_association").isNull(), F.lit("n/a")).otherwise(F.col("author_association"))) \
        .withColumn("user_login", F.when(F.col("user_login").isNull(), F.lit("n/a")).otherwise(F.col("user_login"))) \
        .withColumn("user_type", F.when(F.col("user_type").isNull(), F.lit("n/a")).otherwise(F.col("user_type"))) \
        .withColumn("merge_commit_sha", F.when(F.col("merge_commit_sha").isNull(), F.lit("n/a")).otherwise(F.col("merge_commit_sha"))) \
        .withColumn("base_ref", F.when(F.col("base_ref").isNull(), F.lit("n/a")).otherwise(F.col("base_ref"))) \
        .withColumn("base_sha", F.when(F.col("base_sha").isNull(), F.lit("n/a")).otherwise(F.col("base_sha"))) \
        .withColumn("base_repo_full_name", F.when(F.col("base_repo_full_name").isNull(), F.lit("n/a")).otherwise(F.col("base_repo_full_name"))) \
        .withColumn("head_ref", F.when(F.col("head_ref").isNull(), F.lit("n/a")).otherwise(F.col("head_ref"))) \
        .withColumn("head_sha", F.when(F.col("head_sha").isNull(), F.lit("n/a")).otherwise(F.col("head_sha"))) \
        .withColumn("head_repo_full_name", F.when(F.col("head_repo_full_name").isNull(), F.lit("n/a")).otherwise(F.col("head_repo_full_name"))) \
        .withColumn("milestone_title", F.when(F.col("milestone_title").isNull(), F.lit("n/a")).otherwise(F.col("milestone_title"))) \
        .withColumn("milestone_state", F.when(F.col("milestone_state").isNull(), F.lit("n/a")).otherwise(F.col("milestone_state"))) \
        .withColumn("auto_merge_method", F.when(F.col("auto_merge_method").isNull(), F.lit("n/a")).otherwise(F.col("auto_merge_method"))) \
        .withColumn("pr_id_is_missing", F.when(F.col("pr_id").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("number_is_missing", F.when(F.col("number").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("created_at_is_missing", F.when(F.col("created_at").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("updated_at_is_missing", F.when(F.col("updated_at").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("draft_is_missing", F.when(F.col("draft").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("locked_is_missing", F.when(F.col("locked").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("user_id_is_missing", F.when(F.col("user_id").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("user_site_admin_is_missing", F.when(F.col("user_site_admin").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("closed_at_is_missing", F.when(F.col("closed_at").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("merged_at_is_missing", F.when(F.col("merged_at").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("base_repo_id_is_missing", F.when(F.col("base_repo_id").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("head_repo_id_is_missing", F.when(F.col("head_repo_id").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("milestone_id_is_missing", F.when(F.col("milestone_id").isNull(), F.lit(1)).otherwise(0)) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .dropDuplicates()

        # === Save: silver.<repo>_pull_requests ===
        cleaned_pull_requests.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_pull_requests")
        print(f"Saved: workspace.silver.{repo_name}_pull_requests")

        cleaned_pr_assignees = df \
        .select(
            F.col("id").alias("pr_id"),
            F.explode("assignees").alias("assignee")
        ).select(
            "pr_id",
            F.col("assignee.id").alias("assignee_id"),
            F.trim(F.col("assignee.login")).alias("assignee_login"),
            F.trim(F.col("assignee.type")).alias("assignee_type")
        ) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("pr_id_is_missing", F.when(F.col("pr_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("assignee_id_is_missing", F.when(F.col("assignee_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("assignee_login", F.when(F.col("assignee_login").isNull(), F.lit("n/a")).otherwise(F.col("assignee_login"))) \
        .withColumn("assignee_type", F.when(F.col("assignee_type").isNull(), F.lit("n/a")).otherwise(F.col("assignee_type"))) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .dropDuplicates()

        # === Save: silver.<repo>_pr_assignees ===
        cleaned_pr_assignees.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_pr_assignees")
        print(f"Saved: workspace.silver.{repo_name}_pr_assignees")

        cleaned_pr_labels = df \
        .select(
            F.col("id").alias("pr_id"),
            F.explode("labels").alias("label")
        ).select(
            "pr_id",
            F.col("label.id").alias("label_id"),
            F.upper(F.trim(F.col("label.name"))).alias("label_name"),
            F.trim(F.col("label.color")).alias("label_color")
        ) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("pr_id_is_missing", F.when(F.col("pr_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("label_id_is_missing", F.when(F.col("label_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("label_name", F.when(F.col("label_name").isNull(), F.lit("n/a")).otherwise(F.col("label_name"))) \
        .withColumn("label_color", F.when(F.col("label_color").isNull(), F.lit("n/a")).otherwise(F.col("label_color"))) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .dropDuplicates()

        # === Save: silver.<repo>_pr_labels ===
        cleaned_pr_labels.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_pr_labels")
        print(f"Saved: workspace.silver.{repo_name}_pr_labels")

        cleaned_pr_reviewers = df \
        .select(
            F.col("id").alias("pr_id"),
            F.explode("requested_reviewers").alias("reviewer")
        ).select(
            "pr_id",
            F.col("reviewer.id").alias("reviewer_id"),
            F.trim(F.col("reviewer.login")).alias("reviewer_login"),
            F.trim(F.col("reviewer.type")).alias("reviewer_type")
        ) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("pr_id_is_missing", F.when(F.col("pr_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("reviewer_id_is_missing", F.when(F.col("reviewer_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("reviewer_login", F.when(F.col("reviewer_login").isNull(), F.lit("n/a")).otherwise(F.col("reviewer_login"))) \
        .withColumn("reviewer_type", F.when(F.col("reviewer_type").isNull(), F.lit("n/a")).otherwise(F.col("reviewer_type"))) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .dropDuplicates()

        
        cleaned_pr_reviewers.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_pr_reviewers")
        print(f"Saved: workspace.silver.{repo_name}_pr_reviewers")

        print("=" * 50)

In [0]:
silver_pull_requests()

In [0]:
def silver_contributors():
    endpoints = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
    for endpoint in endpoints:
        repo_name = endpoint
        df = spark.read.table(f"workspace.bronze.raw_{endpoint}_contributors")
        print(endpoint)
        cleaned_contributors = df \
        .select(
            "id",
            F.trim(F.col("login")).alias("login"),
            F.trim(F.col("type")).alias("type"),
            "contributions",
            "site_admin",
        ) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .fillna("n/a") \
        .dropDuplicates()
        cleaned_contributors.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_contributors")
        print(f"Saved: workspace.silver.{repo_name}_contributors")
        print("=" * 50)

In [0]:
silver_contributors()

In [0]:
def silver_languages():
    endpoints = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
    for endpoint in endpoints:
        repo_name = endpoint
        print(endpoint)
        df = spark.read.table(f"workspace.bronze.raw_{endpoint}_languages")
        cleaned_languages = df \
        .select(F.trim(F.col("language")).alias("language"), "bytes") \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .fillna("n/a") \
        .dropDuplicates()
        cleaned_languages.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_languages")
        print(f"Saved: workspace.silver.{repo_name}_languages")
        print("=" * 50)

In [0]:
silver_languages()

In [0]:
def silver_releases():
    endpoints = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
    for endpoint in endpoints:
        repo_name = endpoint
        df = spark.read.table(f"workspace.bronze.raw_{endpoint}_releases")
        print(endpoint)
        cleaned_releases = df \
        .select(
            F.col("id").alias("release_id"),
            F.trim(F.col("tag_name")).alias("tag_name"),
            F.trim(F.col("name")).alias("name"),
            F.col("author.id").alias("author_id"),
            F.col("author.login").alias("author_login"),
            F.col("author.type").alias("author_type"),
            F.col("author.site_admin").alias("author_site_admin"),
            "created_at",
            "published_at",
            "updated_at",
            "draft",
            "prerelease",
            "immutable",
            "target_commitish",
            F.col("reactions.total_count").alias("reactions_total_count"),
            F.col("reactions.`+1`").alias("reactions_plus_one"),
            F.col("reactions.`-1`").alias("reactions_minus_one"),
            F.col("reactions.laugh").alias("reactions_laugh"),
            F.col("reactions.hooray").alias("reactions_hooray"),
            F.col("reactions.confused").alias("reactions_confused"),
            F.col("reactions.heart").alias("reactions_heart"),
            F.col("reactions.rocket").alias("reactions_rocket"),
            F.col("reactions.eyes").alias("reactions_eyes")
        ) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("created_at", F.to_timestamp("created_at")) \
        .withColumn("published_at", F.to_timestamp("published_at")) \
        .withColumn("updated_at", F.to_timestamp("updated_at")) \
        .withColumn("name", F.trim(F.col("name"))) \
        .withColumn(
            "timestamp_issue",
            F.when(F.col("created_at") > F.col("published_at"), F.lit("created_at after published_at"))
            .when(F.col("updated_at") < F.col("published_at"), F.lit("updated_at before published_at"))
            .when(F.col("updated_at") < F.col("created_at"), F.lit("updated_at before created_at"))
            .otherwise(F.lit("n/a"))
        ) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .fillna("n/a") \
        .dropDuplicates()

        cleaned_releases.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{repo_name}_releases")
        print(f"Saved: workspace.silver.{repo_name}_releases")
        print("=" * 50)

In [0]:
silver_releases()

In [0]:
def silver_commits():
    endpoints = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
    for endpoint in endpoints:
        repo_name = endpoint
        df = spark.read.table(f"workspace.bronze.inc_{endpoint}_commits")
        print(endpoint)
        cleaned_commits = df \
        .select(
            F.trim(F.col("sha")).alias("sha"),
            F.trim(F.col("commit.message")).alias("commit_message"),
            F.col("commit.author.date").alias("commit_author_date"),
            F.trim(F.col("commit.author.name")).alias("commit_author_name"),
            F.trim(F.col("commit.author.email")).alias("commit_author_email"),
            F.col("commit.committer.date").alias("commit_committer_date"),
            F.trim(F.col("commit.committer.name")).alias("commit_committer_name"),
            F.trim(F.col("commit.committer.email")).alias("commit_committer_email"),
            F.col("commit.comment_count").alias("comment_count"),
            F.col("author.id").alias("author_id"),
            F.trim(F.col("author.login")).alias("author_login"),
            F.trim(F.col("author.type")).alias("author_type"),
            F.col("author.site_admin").alias("author_site_admin"),
            F.col("committer.id").alias("committer_id"),
            F.trim(F.col("committer.login")).alias("committer_login"),
            F.trim(F.col("committer.type")).alias("committer_type"),
            F.col("committer.site_admin").alias("committer_site_admin")
        ) \
        .withColumn("repo_name", F.lit(repo_name)) \
        .withColumn("commit_author_date", F.to_timestamp(F.col("commit_author_date"))) \
        .withColumn("commit_committer_date", F.to_timestamp(F.col("commit_committer_date"))) \
        .withColumn("sha_is_missing", F.when(F.col("sha").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("commit_message", F.when(F.col("commit_message").isNull(), F.lit("n/a")).otherwise(F.col("commit_message"))) \
        .withColumn("commit_author_date_is_missing", F.when(F.col("commit_author_date").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("commit_author_name", F.when(F.col("commit_author_name").isNull(), F.lit("n/a")).otherwise(F.col("commit_author_name"))) \
        .withColumn("commit_author_email", F.when(F.col("commit_author_email").isNull(), F.lit("n/a")).otherwise(F.col("commit_author_email"))) \
        .withColumn("commit_committer_date_is_missing", F.when(F.col("commit_committer_date").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("commit_committer_name", F.when(F.col("commit_committer_name").isNull(), F.lit("n/a")).otherwise(F.col("commit_committer_name"))) \
        .withColumn("commit_committer_email", F.when(F.col("commit_committer_email").isNull(), F.lit("n/a")).otherwise(F.col("commit_committer_email"))) \
        .withColumn("comment_count_is_missing", F.when(F.col("comment_count").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("author_id_is_missing", F.when(F.col("author_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("author_login", F.when(F.col("author_login").isNull(), F.lit("n/a")).otherwise(F.col("author_login"))) \
        .withColumn("author_type", F.when(F.col("author_type").isNull(), F.lit("n/a")).otherwise(F.col("author_type"))) \
        .withColumn("author_site_admin_is_missing", F.when(F.col("author_site_admin").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("committer_id_is_missing", F.when(F.col("committer_id").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("committer_login", F.when(F.col("committer_login").isNull(), F.lit("n/a")).otherwise(F.col("committer_login"))) \
        .withColumn("committer_type", F.when(F.col("committer_type").isNull(), F.lit("n/a")).otherwise(F.col("committer_type"))) \
        .withColumn("committer_site_admin_is_missing", F.when(F.col("committer_site_admin").isNull(), F.lit(1)).otherwise(F.lit(0))) \
        .withColumn("transformed_at", F.current_timestamp()) \
        .withColumn("commit_author_email", F.when(F.col("commit_author_email") == "", F.lit("n/a")).otherwise(F.col("commit_author_email"))) \
        .withColumn("commit_committer_email", F.when(F.col("commit_committer_email") == "", F.lit("n/a")).otherwise(F.col("commit_committer_email"))) \
        .withColumn("commit_message", F.when(F.col("commit_message") == "", F.lit("n/a")).otherwise(F.col("commit_message"))) \
        .dropDuplicates()
        cleaned_commits.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"workspace.silver.{endpoint}_commits")
        print(f"Saved: workspace.silver.{endpoint}_commits")
        print("=" * 50)

In [0]:
silver_commits()